In [1]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

# Load the cleaned CSV file
df = pd.read_csv("C:/Users/Admin/OneDrive/Desktop/live_headlines.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], errors='coerce')

# Initialize the Dash app
app = Dash(__name__)

# App layout
app.layout = html.Div([
    html.H1("News Headlines Dashboard", style={'textAlign': 'center'}),

    html.Div([
        html.Label("Select Source(s):"),
        dcc.Dropdown(
            id='source-filter',
            options=[{'label': src, 'value': src} for src in sorted(df['source'].dropna().unique())],
            value=sorted(df['source'].dropna().unique()),
            multi=True
        ),
        html.Label("Search by Keyword:"),
        dcc.Input(id='keyword-filter', type='text', placeholder='Enter keyword', style={'width': '100%'})
    ], style={'padding': '10px'}),

    dcc.Graph(id='bar-chart'),
    dcc.Graph(id='time-chart'),
    dcc.Graph(id='clickbait-pie'),
    dcc.Graph(id='sentiment-pie'),

    html.H4("Filtered Headlines Table"),
    html.Div(id='filtered-table')
])

# Callback to update dashboard visuals and table
@app.callback(
    [Output('bar-chart', 'figure'),
     Output('time-chart', 'figure'),
     Output('clickbait-pie', 'figure'),
     Output('sentiment-pie', 'figure'),
     Output('filtered-table', 'children')],
    [Input('source-filter', 'value'),
     Input('keyword-filter', 'value')]
)
def update_dashboard(selected_sources, keyword):
    dff = df[df['source'].isin(selected_sources)]
    if keyword:
        dff = dff[dff['headline'].str.contains(keyword, case=False, na=False)]

    # Bar chart: Headline count by source
    source_counts = dff['source'].value_counts().reset_index()
    source_counts.columns = ['Source', 'Headline Count']
    bar_fig = px.bar(
        source_counts,
        x='Source',
        y='Headline Count',
        title='Headline Count by Source'
    )

    # Line chart: Trend over time
    dff['date'] = dff['timestamp'].dt.date
    trend_df = dff.groupby('date').size().reset_index(name='count')
    time_fig = px.line(
        trend_df,
        x='date',
        y='count',
        title='Headline Trend Over Time'
    )

    # Pie chart: Clickbait vs Non-Clickbait
    cb_df = dff['Clickbait'].value_counts().reset_index()
    cb_df.columns = ['Clickbait', 'Count']
    clickbait_fig = px.pie(cb_df, names='Clickbait', values='Count', title='Clickbait Distribution')

    # Pie chart: Sentiment distribution
    sent_df = dff['Sentiment'].value_counts().reset_index()
    sent_df.columns = ['Sentiment', 'Count']
    sentiment_fig = px.pie(sent_df, names='Sentiment', values='Count', title='Sentiment Distribution')

    # Data table
    table_html = dff[['headline', 'source', 'timestamp', 'Clickbait', 'Sentiment']].to_html(index=False)

    return bar_fig, time_fig, clickbait_fig, sentiment_fig, html.Div([
        dcc.Markdown(f"### {len(dff)} Headlines Found"),
        html.Div(table_html, style={"overflowX": "auto"})
    ])

# Run the Dash app
if __name__ == '__main__':
    app.run(debug=True, port=8052)


C:\Users\Admin\AppData\Local\Temp\ipykernel_25404\69805130.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["timestamp"] = pd.to_datetime(df["timestamp"], errors='coerce')


In [ ]:
LLM Prompt
Took help of llms for solving errors while writing html code in python